<a id="a-context-playbook-evolved"></a>
<a id="ace-improve-an-itemized-playbook"></a>

# ACE: improve an agent's playbook

**What it does.** ACE improves reusable instructions one rule at a time,
preserving useful entries while correcting, retiring, or adding others.

**How it works.** A Generator attempts a task, a Reflector extracts lessons,
and a Curator edits the playbook. Later attempts use the revised rules.

**In this example.** A financial-tagging agent starts with the wrong interest
convention and no lease convention. Follow two revisions, inspect the changed
rules, and check the final playbook on fresh disclosures.

The saved outputs come from deterministic Python agents.
No credentials or model calls are needed. This is a mechanism demonstration
inspired by [ACE](https://arxiv.org/abs/2510.04618), with authored lessons and curation.

<a id="component-worksheet"></a>

## Components of the loop

| Role | Here |
|---|---|
| Artifact | A JSON playbook with an identity, section, rule, and status for each entry |
| Proposer | `revise(lessons)` amends and retires entries, then adds a missing rule |
| Evaluator | `playbook_suite(agent)` checks answers against fixed company conventions |
| Search | Greedy evaluates the seed and tries two revisions on the private split |
| Result | Selected rules, revision history, consolidation checks, and held-out answers |

**Attempt → lesson → edit a rule → check a fresh task.**

[**Download notebook**](https://sentient-xyz.github.io/meta-evolve-docs/downloads/ace-playbook.ipynb)
· [Complete source](https://sentient-xyz.github.io/meta-evolve-docs/downloads/context-example.zip)
· [Display helpers](https://github.com/sentient-xyz/meta-evolve/blob/main/examples/research/context_evolution/ace_display.py)

Seven cells take you through the study and a small exercise. The page and notebook
share the same cells and recorded outputs. Setup downloads the library and a
checksum-verified example archive; subsequent cells run locally.

<a id="setup-and-run"></a>

## 1. Prepare the example

Use a Python 3.12+ Jupyter kernel on macOS or Linux. Run these cells in order,
or copy them into an empty notebook. Setup needs internet access; a checkout
and companion-file uploads are unnecessary.

In [1]:
%pip install -q https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-evolve.zip
from hashlib import sha256
from io import BytesIO
from pathlib import Path
import sys, tempfile
from urllib.request import urlopen
from zipfile import ZipFile

source_url = "https://sentient-xyz.github.io/meta-evolve-docs/downloads/context/92c3688fd94dc264d0bf0be2f3ce8f7690ce463799c86467bf714da2d2a1f443/example.zip"
archive = urlopen(source_url, timeout=30).read()
if sha256(archive).hexdigest() != "92c3688fd94dc264d0bf0be2f3ce8f7690ce463799c86467bf714da2d2a1f443":
    raise RuntimeError("Example download does not match this notebook.")
support = Path(tempfile.mkdtemp(prefix="ace-playbook-"))
with ZipFile(BytesIO(archive)) as bundle:
    bundle.extractall(support)
sys.path.insert(0, str(support / "examples/research/context_evolution"))

from IPython.display import HTML, display
from ace_context import PLAYBOOK_CALL_CEILING, PLAYBOOK_FIXTURE_AGENT, SEED_PLAYBOOK
from ace_study import study_playbook
from ace_display import answers, changes, consolidation_view, entries, lessons_view, revisions

print("Ready · deterministic fixture · no model calls")

Ready · deterministic fixture · no model calls


<a id="the-concrete-problem"></a>

## 2. Read the starting playbook

Northwind uses its own taxonomy elements for cash interest and operating leases.
The starting manual names the generic interest element and an unsupported scaling
rule. It says nothing about leases. These are the instructions we will improve;
the agent and the grading rules stay fixed.

In [2]:
display(HTML(entries(SEED_PLAYBOOK)))

<a id="follow-the-two-levels"></a>
<a id="the-actual-experiment"></a>
<a id="what-that-one-call-did"></a>
<a id="three-splits-two-tagging-policies"></a>

## 3. Learn from the missed answers

`study_playbook` runs **development → private search → consolidation → held out**
once, in that order. Its [search declaration](https://github.com/sentient-xyz/meta-evolve/blob/main/examples/research/context_evolution/ace_search.py)
uses an ordinary `meta.Experiment` with a bounded Greedy policy. The remaining
displays read this study; they do not rerun phases.

First, inspect the seed's two development answers. The fixture follows the
conventions it is given. Python publishes the authored lesson for each completed
wrong answer; only these development lessons reach the curator.

In [3]:
study = study_playbook(PLAYBOOK_FIXTURE_AGENT)
display(HTML(answers(study.development)))
display(HTML(lessons_view(study.lessons)))

<a id="the-revision-mechanism"></a>

## 4. Follow the two revisions

The first proposal amends the interest rule and retires the unsupported scale
entry. The next adds the lease rule. Each version answers two differently worded
**private** disclosures; Greedy keeps the best measured playbook and stops at 100%.
Private answers and held-out tasks never enter the published development lessons.

The table reports those private scores. The diffs below it show the actual rule
text, keyed by the same entry identities throughout the run.

In [4]:
display(HTML(revisions(study.run)))

Version,Private solved,Result
Seed,0%,Evaluated
Revision 1,50%,Evaluated
Revision 2,100%,Selected


## 5. Keep a smaller playbook

Retirement stops an entry being included in the agent's instructions while keeping
it in the candidate's record. After selection, consolidation proposes removing
retired entries and independently checks both versions on the private split.
Here the checks are neutral, so the pruned playbook is accepted. Earlier versions
still contain the retired entry.

In [5]:
display(HTML(consolidation_view(study.consolidation)))
display(HTML(entries(study.playbook)))

Matched private check,Solved
Selected playbook,100%
Pruned playbook,100%


## 6. Check fresh disclosures

The accepted playbook now tags the held-out **borrowings** and **right-of-use**
notes. These tasks were evaluated only after selection and consolidation.
This reports final behavior on two authored cases; it is not a held-out
before/after comparison or a claim about benchmark gains.

In [6]:
display(HTML(answers(study.held_out)))
print(f"All phases: {study.sessions}/{PLAYBOOK_CALL_CEILING} fixture calls")
print(f"Search: {study.run.usage().evaluations} evaluations · {study.run.usage().trials} revisions")
print(f"Reported usage: {study.usage.tokens} tokens · {study.usage.spend_micros} spend_micros")
print(f"Retained failures: {len(study.failures)}")

All phases: 14/14 fixture calls
Search: 3 evaluations · 2 revisions
Reported usage: 0 tokens · 0 spend_micros
Retained failures: 0


`study.usage` covers all four phases; `study.run.usage()` covers search only.
The fixture reports zero model usage. Its fourteen calls include development,
private search, both consolidation checks, and held-out work.

<a id="change-and-predict"></a>

## 7. Change a lesson and predict the edit

Give the curator a different interest convention. Before running the cell,
predict whether it amends the existing entry or appends a conflicting one.
This is a local proposal exercise: it shows the edit without evaluating or
replacing the study's selected playbook.

In [7]:
from ace_curation import revise
from ace_rollouts import Lesson
from playbook_codec import decode, encode

lesson = Lesson(
    identity="interest-element", section="debt", marker="nwi:InterestPaidGross",
    content="Cash interest is tagged nwi:InterestPaidGross, before capitalization.",
)
proposal = revise((lesson,))(encode(SEED_PLAYBOOK), context=None)
if proposal.failure:
    print(proposal.failure)
else:
    display(HTML(changes(SEED_PLAYBOOK, decode(proposal.candidate))))

<a id="what-the-paper-does"></a>
<a id="evidence-and-limitations"></a>

## Reading the evidence

The saved private scores progress from **0% → 50% → 100%**. The corrected
interest entry keeps its identity, the scale entry is retired before being
pruned, and a new lease entry supplies the missing convention.

The [paper's Generator / Reflector / Curator system](https://arxiv.org/html/2510.04618v3#S3)
uses learned reflection and incremental context updates. Here Python supplies
the lessons and edits. Learned reflection, semantic retrieval/deduplication,
helpful/harmful counters, and online adaptation are outside this demonstration.

<a id="why-identity-is-the-whole-point"></a>

<details markdown="1"><summary>Identity, authority, and saved runs</summary>

Entries retain their logical identities across amendments and retirement.
Canonical JSON gives equal playbooks equal bytes. The curator declares its
development lessons; the evaluator declares its fixed agent, split membership,
and budgets. Candidates control only playbook content.

The notebook uses in-memory storage. Passing `storage=meta.Storage.durable(path)`
to `study_playbook` saves the **outer search** for reopening. Development,
consolidation, and held-out outcomes remain on the returned study; they are
not persisted or resumable phases. See the
[storage contract](https://sentient-xyz.github.io/meta-evolve-docs/guides/durability/).

</details>

<a id="four-ways-to-be-unrankable"></a>
<a id="a-split-that-did-not-finish-has-no-score"></a>

<details markdown="1"><summary>Failures remain different from wrong answers</summary>

| Outcome | Meaning |
|---|---|
| `invalid` | The candidate is not a valid playbook or transition |
| `wrapper-failed` | The agent did not produce a usable answer |
| `unverifiable` | The target needed for checking is absent |
| `infrastructure-failed` | A timeout or infrastructure fault prevented measurement |

A completed wrong answer scores zero. These failures carry no rankable score.
An incomplete split refuses an aggregate rather than averaging its successful
rollouts. All phases retain their failures and reported usage; focused
[failure tests](https://github.com/sentient-xyz/meta-evolve/blob/main/tests/test_context_evolution_ace_failures.py)
exercise these boundaries.

</details>

<a id="complete-supporting-source"></a>

For the script, run `uv run python examples/research/context_evolution/main.py --sibling ace`
from a checkout. The [example README](https://github.com/sentient-xyz/meta-evolve/blob/main/examples/research/context_evolution/README.md)
covers the source layout, fixture command, and `InnerAgent` boundary for your own SDK.
[Browse the source](https://github.com/sentient-xyz/meta-evolve/tree/main/examples/research/context_evolution)
to inspect each stage.

For a smaller introduction, [improve a support agent's playbook](https://sentient-xyz.github.io/meta-evolve-docs/guides/playbook-improvement/)
with six tickets and one revision per recorded mistake.

[EvoSkill notebook](https://github.com/sentient-xyz/meta-evolve/blob/main/docs/notebooks/context_evolution/evoskill.ipynb)
· [Feedback Descent](https://sentient-xyz.github.io/meta-evolve-docs/build-patterns/feedback-descent/)
· [Explore examples](https://sentient-xyz.github.io/meta-evolve-docs/examples/)